# 🐍 Módulo: Estructuras de control
## RA5 · Excepciones y código robusto

**Resultado de aprendizaje:** *Crea arquitectura de código de forma eficiente y escribe código robusto.*

### Criterios de evaluación que se trabajan

| Criterio | Descripción | Apartado |
|---|---|---|
| a | Diferenciar entre excepción y error de sintaxis | 1 |
| b | Escribir instrucciones de captura de excepciones | 2 |
| c | Capturar y tratar excepciones | 3 |
| d | Tratar excepciones (lanzar, propagar, personalizar) | 4 |
| e | Realizar depuraciones de excepciones correctamente | 5 |
| f | Escribir bloques de código robusto con las sentencias adecuadas | 6 |

## 1. Excepciones frente a errores de sintaxis (criterio a)

Hay dos grandes momentos en los que algo puede fallar:

| | Error de **sintaxis** | **Excepción** |
|---|---|---|
| ¿Cuándo se detecta? | **Antes** de ejecutar (al compilar/analizar el código) | **Durante** la ejecución |
| Causa | El código no respeta la gramática de Python | El código es válido pero ocurre algo imprevisto (dato erróneo, fichero inexistente...) |
| ¿Se ejecuta parte del programa? | **No**, no se ejecuta nada | Sí, hasta que se produce la excepción |
| ¿Se puede capturar? | No directamente con `try` en el mismo archivo | **Sí**, con `try/except` |
| Ejemplos | `SyntaxError`, `IndentationError` | `ZeroDivisionError`, `ValueError`, `TypeError`... |

In [ ]:
# Error de sintaxis: Python ni siquiera empieza a ejecutar este código.
# Lo comprobamos con compile(), que solo analiza el código sin ejecutarlo.
codigo_malo = "print('hola'"           # falta cerrar el paréntesis

try:
    compile(codigo_malo, "<ejemplo>", "exec")
except SyntaxError as e:
    print("Error de sintaxis detectado ANTES de ejecutar:")
    print("  ", type(e).__name__, "->", e.msg)

In [ ]:
# Excepción: el código es sintácticamente correcto, pero falla al ejecutarse
print("Esta línea SÍ se ejecuta")
try:
    resultado = 10 / 0
except ZeroDivisionError as e:
    print("Excepción en tiempo de ejecución:", e)

In [ ]:
# Un código con sintaxis válida pero con errores lógicos o de ejecución
casos = {
    "ZeroDivisionError": lambda: 1 / 0,
    "ValueError": lambda: int("abc"),
    "TypeError": lambda: "2" + 2,
    "IndexError": lambda: [1, 2, 3][10],
    "KeyError": lambda: {"a": 1}["b"],
    "NameError": lambda: variable_inexistente,
    "AttributeError": lambda: "texto".no_existe(),
    "FileNotFoundError": lambda: open("no_existe.txt"),
    "ModuleNotFoundError": lambda: __import__("modulo_que_no_existe"),
}

for nombre, accion in casos.items():
    try:
        accion()
    except Exception as e:
        print(f"{type(e).__name__:20} -> {e}")

### 1.1 Jerarquía de excepciones (resumen)

```
BaseException
 ├── KeyboardInterrupt
 ├── SystemExit
 └── Exception                  <- de aquí cuelgan casi todas
      ├── ArithmeticError  ── ZeroDivisionError, OverflowError
      ├── LookupError      ── IndexError, KeyError
      ├── OSError          ── FileNotFoundError, PermissionError
      ├── ValueError
      ├── TypeError
      ├── NameError
      ├── AttributeError
      └── RuntimeError     ── RecursionError
```

Capturar una clase captura también todas sus **subclases**.

In [ ]:
print(ZeroDivisionError.__mro__)
print(issubclass(KeyError, LookupError), issubclass(FileNotFoundError, OSError))

### 1.2 Leer un *traceback*

Cuando una excepción no se captura, Python muestra la **traza** (de la llamada más antigua a la más reciente). Lo más importante: **la última línea** (tipo y mensaje) y la línea de tu código donde ocurrió.

In [ ]:
import traceback

def a():
    return b()

def b():
    return 1 / 0

try:
    a()
except ZeroDivisionError:
    traceback.print_exc()     # muestra la traza sin detener el cuaderno

### ✏️ Actividad 1
Clasifica cada caso como **error de sintaxis** o **excepción** y, si es excepción, indica su tipo:

1. `print("Hola)`
2. `int("3.5")`
3. `lista = [1, 2]; lista[5]`
4. `if x = 3:`
5. `3 + "3"`
6. `for i in range(3)` (sin dos puntos)
7. `d = {}; d["x"]`

In [ ]:
respuestas = {
    1: "",  # TU RESPUESTA
    2: "",
    3: "",
    4: "",
    5: "",
    6: "",
    7: "",
}
print(respuestas)

## 2. Instrucciones de captura de excepciones (criterio b)

```python
try:
    # código que puede fallar
except TipoDeExcepcion:
    # qué hacer si falla
else:
    # se ejecuta si NO hubo excepción
finally:
    # se ejecuta SIEMPRE
```

```mermaid
flowchart TD
    A[try] --> B{¿Excepción?}
    B -- No --> C[else]
    B -- Sí --> D[except que coincida]
    C --> E[finally]
    D --> E
    E --> F[Continúa el programa]
```

In [ ]:
# try / except básico
try:
    numero = int("42")
    print("Convertido:", numero)
except ValueError:
    print("Eso no es un número")

try:
    numero = int("cuarenta y dos")
    print("Convertido:", numero)
except ValueError:
    print("Eso no es un número")

print("El programa continúa")

In [ ]:
# Sin captura, el programa se detendría. Con captura, se recupera.
def dividir(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("No se puede dividir entre cero")
        return None

print(dividir(10, 2))
print(dividir(10, 0))

In [ ]:
# else y finally
def leer_numero(texto):
    try:
        n = int(texto)
    except ValueError:
        print(f"  '{texto}' no es válido")
        return None
    else:
        print(f"  '{texto}' convertido correctamente")   # solo si no hubo excepción
        return n
    finally:
        print("  (fin del intento)")                     # siempre

leer_numero("7")
leer_numero("x")

### 2.1 Varias excepciones

In [ ]:
def operar(lista, indice, divisor):
    try:
        valor = lista[indice]
        return valor / divisor
    except IndexError:
        return "Índice fuera de rango"
    except ZeroDivisionError:
        return "División por cero"
    except TypeError:
        return "Tipos incompatibles"

print(operar([10, 20], 1, 5))
print(operar([10, 20], 5, 5))
print(operar([10, 20], 1, 0))
print(operar([10, "a"], 1, 2))

In [ ]:
# Agrupar varias excepciones en una sola cláusula
for dato in ["12", "abc", None, 3.7]:
    try:
        print(int(dato), end=" | ")
    except (ValueError, TypeError) as e:
        print(f"{type(e).__name__}", end=" | ")
print()

> ⚠️ El **orden** importa: se ejecuta el **primer** `except` que coincida. Pon primero las excepciones **específicas** y al final las generales.

In [ ]:
try:
    {}["x"]
except LookupError:
    print("Capturada como LookupError (clase padre)")
except KeyError:
    print("Esto NUNCA se ejecutará (KeyError ya está cubierta por LookupError)")

### ✏️ Actividad 2
Escribe una función `convertir_a_entero(valor)` que devuelva el entero correspondiente o `None` si no es posible. Debe funcionar con `"12"`, `"abc"`, `None`, `7.9`, `"3.5"`.

In [ ]:
def convertir_a_entero(valor):
    # TU CÓDIGO AQUÍ
    pass

for v in ["12", "abc", None, 7.9, "3.5"]:
    print(repr(v), "->", convertir_a_entero(v))

## 3. Capturar y tratar excepciones (criterio c)

Capturar no basta: hay que **tratar** la excepción con una estrategia clara.

| Estrategia | Ejemplo |
|---|---|
| **Recuperarse** con un valor por defecto | Usar 0 si el dato no es válido |
| **Reintentar** | Volver a pedir un dato |
| **Informar** al usuario/registrar y continuar | Mensaje claro, log |
| **Limpiar recursos** y propagar | `finally`, `with` |
| **Abortar** de forma controlada | Mensaje y salida |

### 3.1 Obtener información de la excepción

In [ ]:
try:
    int("hola")
except ValueError as e:
    print("Tipo   :", type(e).__name__)
    print("Mensaje:", e)
    print("Args   :", e.args)

In [ ]:
# Estrategia 1: valor por defecto
def a_float(texto, por_defecto=0.0):
    try:
        return float(texto)
    except ValueError:
        return por_defecto

print(a_float("3.14"), a_float("n/a"), a_float("n/a", -1))

In [ ]:
# Estrategia 2: reintentar hasta obtener un dato válido (entradas simuladas)
intentos = iter(["abc", "-5", "x", "25"])

def pedir_edad():
    while True:
        texto = next(intentos)                # en un programa real: input("Edad: ")
        try:
            edad = int(texto)
            if edad < 0:
                raise ValueError("la edad no puede ser negativa")
        except ValueError as e:
            print(f"  Entrada '{texto}' no válida ({e}). Inténtalo de nuevo.")
        else:
            return edad

print("Edad:", pedir_edad())

In [ ]:
# Estrategia 3: tratar el error y seguir con el resto de datos
registros = ["10", "20", "x", "40", "", "50"]
validos = []
errores = []

for r in registros:
    try:
        validos.append(int(r))
    except ValueError:
        errores.append(r)

print("Válidos:", validos)
print("Errores:", errores)
print("Suma de los válidos:", sum(validos))

In [ ]:
# Estrategia 4: ficheros y limpieza de recursos
def leer_primera_linea(ruta):
    try:
        with open(ruta, encoding="utf-8") as f:      # 'with' cierra el fichero siempre
            return f.readline().strip()
    except FileNotFoundError:
        return f"El fichero '{ruta}' no existe"
    except PermissionError:
        return f"Sin permisos para leer '{ruta}'"

print(leer_primera_linea("no_existe.txt"))

# Creamos un fichero de prueba y lo leemos
with open("prueba.txt", "w", encoding="utf-8") as f:
    f.write("Primera línea de prueba\nSegunda\n")
print(leer_primera_linea("prueba.txt"))

### 3.2 Lo que NO hay que hacer

In [ ]:
# ❌ Mal: 'except' desnudo que oculta cualquier error (incluso Ctrl+C)
try:
    resultado = 10 / 0
except:
    pass          # el error desaparece sin dejar rastro

# ❌ Mal: capturar Exception y no hacer nada útil
# ✅ Bien: capturar lo más específico posible y tratar / registrar el error
try:
    resultado = 10 / 0
except ZeroDivisionError as e:
    print("Aviso: división por cero; se usa 0 como resultado")
    resultado = 0

### ✏️ Actividad 3
Dada la lista de cadenas `datos = ["3", "x", "8", "", "5.5", "10"]`:

1. Convierte cada elemento a `float` usando `try/except` y calcula la **media** de los válidos.
2. Cuenta cuántos fallaron y de qué **tipo** de excepción.
3. Si no hay ningún valor válido, la media debe ser `None` (sin lanzar excepción).

In [ ]:
datos = ["3", "x", "8", "", "5.5", "10"]
# TU CÓDIGO AQUÍ

## 4. Tratamiento avanzado: lanzar, relanzar y crear excepciones (criterio d)

### 4.1 Lanzar excepciones con `raise`

Tú también puedes **provocar** una excepción cuando detectas un problema (validación).

In [ ]:
def calcular_raiz(n):
    if not isinstance(n, (int, float)):
        raise TypeError("n debe ser numérico")
    if n < 0:
        raise ValueError("n no puede ser negativo")
    return n ** 0.5

for valor in (16, -4, "a"):
    try:
        print(calcular_raiz(valor))
    except (ValueError, TypeError) as e:
        print(f"{type(e).__name__}: {e}")

In [ ]:
# Propagación: si nadie la captura, la excepción 'sube' por la pila de llamadas
def nivel3():
    raise RuntimeError("fallo en nivel 3")

def nivel2():
    nivel3()

def nivel1():
    try:
        nivel2()
    except RuntimeError as e:
        print("Capturada en nivel1:", e)

nivel1()

In [ ]:
# Relanzar (raise sin argumentos) tras hacer algo, y encadenar (raise ... from ...)
def procesar(dato):
    try:
        return int(dato)
    except ValueError as e:
        print(f"  (registro) dato problemático: {dato!r}")
        raise                      # vuelve a lanzar la misma excepción

try:
    procesar("x")
except ValueError:
    print("Capturada fuera tras relanzar")

def cargar_edad(texto):
    try:
        return int(texto)
    except ValueError as e:
        raise ValueError(f"Edad no válida: {texto!r}") from e

try:
    cargar_edad("veinte")
except ValueError as e:
    print(e, "| causa original:", repr(e.__cause__))

### 4.2 Excepciones personalizadas

Crear tus propias excepciones hace el código más **expresivo** y permite capturar errores de tu dominio.

In [ ]:
class SaldoInsuficienteError(Exception):
    """Se lanza cuando se intenta retirar más dinero del disponible."""
    def __init__(self, saldo, importe):
        self.saldo = saldo
        self.importe = importe
        super().__init__(f"Saldo {saldo} € insuficiente para retirar {importe} €")

class CuentaBloqueadaError(Exception):
    pass

class Cuenta:
    def __init__(self, saldo, bloqueada=False):
        self.saldo = saldo
        self.bloqueada = bloqueada

    def retirar(self, importe):
        if self.bloqueada:
            raise CuentaBloqueadaError("La cuenta está bloqueada")
        if importe <= 0:
            raise ValueError("El importe debe ser positivo")
        if importe > self.saldo:
            raise SaldoInsuficienteError(self.saldo, importe)
        self.saldo -= importe
        return self.saldo

cuenta = Cuenta(100)
for importe in (30, 100, -5):
    try:
        print("Nuevo saldo:", cuenta.retirar(importe))
    except SaldoInsuficienteError as e:
        print("Error:", e, "| faltan", e.importe - e.saldo, "€")
    except ValueError as e:
        print("Error de valor:", e)

try:
    Cuenta(50, bloqueada=True).retirar(10)
except CuentaBloqueadaError as e:
    print("Error:", e)

### 4.3 `finally` y recursos

`finally` garantiza la limpieza incluso si hay `return` o una excepción no capturada.

In [ ]:
def con_finally():
    try:
        print("  trabajando...")
        return "resultado"
    finally:
        print("  limpieza (siempre se ejecuta)")

print(con_finally())

In [ ]:
# Gestor de contexto (with): la forma elegante de garantizar la limpieza
class Recurso:
    def __enter__(self):
        print("  recurso abierto")
        return self
    def __exit__(self, tipo, valor, traza):
        print("  recurso cerrado")
        return False         # no suprime la excepción

try:
    with Recurso():
        print("  usando recurso")
        raise ValueError("problema mientras se usaba")
except ValueError as e:
    print("Capturada:", e)

### ✏️ Actividad 4
1. Crea una función `validar_edad(edad)` que lance `TypeError` si no es entero y `ValueError` si no está entre 0 y 120. Pruébala con varios valores.
2. Crea una excepción personalizada `StockInsuficienteError` y una función `vender(stock, cantidad)` que la lance si no hay unidades suficientes.

In [ ]:
# TU CÓDIGO AQUÍ

## 5. Depuración de excepciones (criterio e)

**Depurar** (*debugging*) es localizar y corregir la causa de un error. Método:

1. **Reproducir** el error con un caso mínimo.
2. **Leer el traceback** (última línea = qué; líneas anteriores = dónde).
3. **Formular hipótesis** y comprobarlas.
4. **Corregir** y **volver a probar** (idealmente con un test).

### 5.1 Herramientas

| Herramienta | Uso |
|---|---|
| `print()` / f-strings con `!r` | Ver el estado de las variables |
| `traceback` | Obtener/mostrar la traza de una excepción capturada |
| `logging` | Registro profesional con niveles (DEBUG, INFO, WARNING, ERROR) |
| `assert` | Comprobar condiciones que **deben** cumplirse |
| `breakpoint()` / `pdb` | Depurador interactivo línea a línea |
| Depurador del IDE (VS Code, PyCharm) | Puntos de ruptura, inspección de variables |

In [ ]:
# Depuración con prints: localizar un fallo
def media_notas(notas):
    total = 0
    for nota in notas:
        print(f"DEBUG nota={nota!r} tipo={type(nota).__name__}")   # ← pista
        total += nota
    return total / len(notas)

try:
    media_notas([7, 8, "9"])
except TypeError as e:
    print("TypeError:", e)
    print("→ Hipótesis: hay una cadena mezclada con números.")

In [ ]:
# Corrección: convertir los datos y tratar los no válidos
def media_notas_robusta(notas):
    validas = []
    for nota in notas:
        try:
            validas.append(float(nota))
        except (ValueError, TypeError):
            print(f"  Nota ignorada: {nota!r}")
    if not validas:
        raise ValueError("No hay notas válidas")
    return sum(validas) / len(validas)

print(media_notas_robusta([7, 8, "9", "x", None]))

In [ ]:
# traceback: obtener información detallada de una excepción capturada
import traceback, sys

def falla():
    lista = [1, 2, 3]
    return lista[10]

try:
    falla()
except IndexError as e:
    tipo, valor, tb = sys.exc_info()
    print("Tipo   :", tipo.__name__)
    print("Mensaje:", valor)
    ultimo = traceback.extract_tb(tb)[-1]
    print(f"Línea {ultimo.lineno} en '{ultimo.name}': {ultimo.line}")

In [ ]:
# logging: registro en lugar de prints
import logging, io

flujo = io.StringIO()                 # para mostrar aquí el log (normalmente va a un fichero o consola)
logging.basicConfig(stream=flujo, level=logging.DEBUG,
                    format="%(levelname)s - %(funcName)s - %(message)s", force=True)

def dividir_log(a, b):
    logging.debug("Dividiendo %s entre %s", a, b)
    try:
        return a / b
    except ZeroDivisionError:
        logging.exception("Error al dividir")        # incluye la traza
        return None

dividir_log(10, 2)
dividir_log(1, 0)
print(flujo.getvalue())

In [ ]:
# assert: comprobar invariantes durante el desarrollo
def aplicar_descuento(precio, porcentaje):
    assert precio >= 0, f"precio negativo: {precio}"
    assert 0 <= porcentaje <= 100, f"porcentaje fuera de rango: {porcentaje}"
    return precio * (1 - porcentaje / 100)

try:
    aplicar_descuento(100, 150)
except AssertionError as e:
    print("AssertionError:", e)

print(aplicar_descuento(100, 20))

### 5.2 Depurador interactivo (`pdb`)

Pon `breakpoint()` donde quieras parar la ejecución. En el prompt `(Pdb)` puedes usar:

| Comando | Acción |
|---|---|
| `n` | Siguiente línea (*next*) |
| `s` | Entrar en una función (*step*) |
| `c` | Continuar hasta el siguiente punto de ruptura |
| `p variable` | Mostrar una variable |
| `l` | Mostrar el código alrededor |
| `q` | Salir |

```python
def ejemplo(x):
    y = x * 2
    breakpoint()      # <- se detiene aquí (no lo ejecutamos en este cuaderno)
    return y / 0
```

En un cuaderno Jupyter también puedes usar la *magia* `%debug` justo después de una excepción para depurar post-mortem.

### 5.3 Ejercicio de depuración

El código siguiente contiene **varios errores**. Encuéntralos usando lo aprendido (lee el mensaje, añade `print`, corrige).

In [ ]:
def resumen_ventas(ventas):
    """ventas: lista de diccionarios {'producto': str, 'unidades': int, 'precio': float}"""
    total = 0
    for v in ventas:
        total += v["unidades"] * v["precio"]
    return {"total": total, "media": total / len(ventas)}

ventas = [
    {"producto": "A", "unidades": 3, "precio": 9.5},
    {"producto": "B", "unidades": "2", "precio": 4.0},     # ← error de tipo
    {"producto": "C", "precio": 7.0},                      # ← falta clave
]

try:
    print(resumen_ventas(ventas))
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# TU CÓDIGO AQUÍ: escribe resumen_ventas_robusto(ventas) que ignore y cuente los registros erróneos
# y devuelva también la lista de errores encontrados. Maneja el caso de lista vacía.

## 6. Código robusto (criterio f)

**Código robusto**: sigue funcionando (o falla de forma controlada) ante entradas inesperadas, y nunca deja recursos sin liberar ni datos corruptos.

### 6.1 Principios

1. **Validar** las entradas (tipo, rango, formato).
2. **Capturar solo lo necesario** y lo más específico posible.
3. **Mensajes de error claros** que indiquen qué pasó y cómo solucionarlo.
4. **Liberar recursos** (`with`, `finally`).
5. **Fallar pronto** (cláusulas de guarda) y no ocultar errores.
6. **Registrar** (`logging`) lo ocurrido.
7. **Probar** casos normales, límite y erróneos.

### 6.2 LBYL frente a EAFP

| Estilo | Significado | Ejemplo |
|---|---|---|
| **LBYL** | *Look Before You Leap*: comprobar antes | `if clave in dic:` |
| **EAFP** | *Easier to Ask Forgiveness than Permission*: intentar y capturar (más pythónico) | `try: dic[clave] except KeyError:` |

In [ ]:
dic = {"a": 1}

# LBYL
if "b" in dic:
    valor = dic["b"]
else:
    valor = 0

# EAFP
try:
    valor = dic["b"]
except KeyError:
    valor = 0

# Para diccionarios hay una opción aún más simple
valor = dic.get("b", 0)
print(valor)

In [ ]:
# Función robusta: validación + excepciones específicas + mensajes claros
def dividir_seguro(a, b):
    """Devuelve a/b. Lanza TypeError/ValueError con mensajes claros."""
    for nombre, valor in (("a", a), ("b", b)):
        if isinstance(valor, bool) or not isinstance(valor, (int, float)):
            raise TypeError(f"'{nombre}' debe ser numérico, no {type(valor).__name__}")
    if b == 0:
        raise ValueError("El divisor no puede ser cero")
    return a / b

for args in [(10, 4), (1, 0), ("1", 2), (5, None)]:
    try:
        print(args, "->", dividir_seguro(*args))
    except (TypeError, ValueError) as e:
        print(args, "-> Error:", e)

In [ ]:
# Programa robusto completo: lectura de un fichero CSV sencillo con datos erróneos
contenido = """ana,8.5
luis,abc
marta,9
,7
pedro,4.5,extra
"""
with open("notas.csv", "w", encoding="utf-8") as f:
    f.write(contenido)

def cargar_notas(ruta):
    notas, errores = {}, []
    try:
        with open(ruta, encoding="utf-8") as f:
            for n_linea, linea in enumerate(f, start=1):
                try:
                    nombre, nota = linea.strip().split(",")
                    if not nombre:
                        raise ValueError("nombre vacío")
                    notas[nombre] = float(nota)
                except ValueError as e:
                    errores.append((n_linea, linea.strip(), str(e)))
    except FileNotFoundError:
        raise FileNotFoundError(f"No se encontró '{ruta}'") from None
    return notas, errores

notas, errores = cargar_notas("notas.csv")
print("Notas válidas:", notas)
print("Errores:")
for e in errores:
    print("  línea", e[0], "->", e[1], "|", e[2])

In [ ]:
# Robustez ante recursos: reintentos con espera creciente (simulado)
import time

def operacion_inestable(intento):
    if intento < 3:
        raise ConnectionError(f"fallo de red en el intento {intento}")
    return "datos recibidos"

def con_reintentos(funcion, maximo=5):
    for intento in range(1, maximo + 1):
        try:
            return funcion(intento)
        except ConnectionError as e:
            print(f"  {e}")
            if intento == maximo:
                raise                      # agotados los reintentos: propaga
            time.sleep(0.05 * intento)     # espera creciente

print(con_reintentos(operacion_inestable))

### ✏️ Actividades finales

**A.** Escribe `leer_enteros(ruta)` que lea un fichero con un número por línea, ignore las líneas inválidas y devuelva la lista de enteros. Debe manejar `FileNotFoundError`.

**B.** Escribe una **calculadora robusta** `calcular(expresion)` que reciba textos como `"8 / 2"` o `"3 * x"` y devuelva el resultado o un mensaje de error claro para: operador no soportado, operandos no numéricos, división por cero, formato incorrecto.

**C.** Crea una clase `Inventario` con una excepción propia `ProductoNoEncontradoError` y métodos `añadir`, `retirar` y `consultar` que validen sus entradas.

In [ ]:
# A
# TU CÓDIGO AQUÍ

In [ ]:
# B
# TU CÓDIGO AQUÍ

In [ ]:
# C
# TU CÓDIGO AQUÍ

## ✅ Autoevaluación

- [ ] **a)** Distingo un error de sintaxis (antes de ejecutar) de una excepción (en ejecución).
- [ ] **b)** Escribo `try`, `except`, `else` y `finally` y sé en qué orden se ejecutan.
- [ ] **c)** Capturo excepciones específicas y las trato (valor por defecto, reintento, aviso).
- [ ] **d)** Lanzo excepciones con `raise`, las relanzo y creo excepciones personalizadas.
- [ ] **e)** Sé leer un *traceback* y depurar con `print`, `logging`, `assert` y `pdb`.
- [ ] **f)** Escribo código robusto: valido entradas, libero recursos con `with`/`finally` y fallo con mensajes claros.

🎉 **Has completado el módulo de Estructuras de control.**